# Notebook 07: Power BI Data Preparation & Star Schema Engineering

**Project:** UCI Online Retail II E-Commerce Sales Analytics & Business Intelligence  
**Phase:** Power BI Data Preparation & Semantic Modeling  
**Primary Source:** `data/processed/ecommerce_analytics.db` (SQLite Analytics Warehouse)  
**Output Destination:** `data/processed/powerbi/` & SQLite Dimensional Tables  

---

## Executive Overview & Architectural Objectives

This notebook engineers an enterprise-grade **Kimball Star Schema** to serve as the unified data foundation for the Power BI Business Intelligence dashboard. 

### Core Architectural Principles
1. **Decoupled Dimensional Architecture:** Separate business process event records (Facts: Sales, Returns) from context-providing entities (Dimensions: Time, Customer, Product, Geography).
2. **Single-Direction Filter Flow:** All dimensional relationships are established as 1-to-many (`1:*`) with single-direction filtering (Dimension $\rightarrow$ Fact) to maximize VertiPaq query engine performance and eliminate ambiguous filter propagation.
3. **Shared Conforming Dimensions:** Both `fact_sales` and `fact_returns` share identical dimensional grain across Time, Customer, Product, and Geography, enabling synchronized slicing and accurate net sales calculations.
4. **Zero-Orphan Key Guarantee:** Full referential integrity audit ensures that 100% of foreign keys in fact tables find matching primary keys in dimension tables, cleanly handling edge cases (guest checkouts and return-only customers).
5. **Storage & Performance Optimization:** Eliminate denormalized calendar attributes from transaction tables and export lean, typed schemas ready for instant ingestion in Power BI Desktop.


In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
from decimal import Decimal

# Configure pandas display options
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# Output directory for Power BI datasets
pbi_dir = os.path.join('..', 'data', 'processed', 'powerbi')
os.makedirs(pbi_dir, exist_ok=True)

# Database path
db_path = os.path.join('..', 'data', 'processed', 'ecommerce_analytics.db')

print(f"[OK] Environment configured.")
print(f"[OK] Database connection path: {os.path.abspath(db_path)}")
print(f"[OK] Power BI export directory: {os.path.abspath(pbi_dir)}")


[OK] Environment configured.
[OK] Database connection path: C:\Users\sabha\OneDrive\Desktop\Freelance\Ecommerce-Sales-Analytics\data\processed\ecommerce_analytics.db
[OK] Power BI export directory: C:\Users\sabha\OneDrive\Desktop\Freelance\Ecommerce-Sales-Analytics\data\processed\powerbi


## 1. Source Database Audit & Benchmark Validation

Before dimensional engineering, we query `data/processed/ecommerce_analytics.db` to verify baseline commercial KPIs against the reconciled benchmarks established in Notebook 06 and the Metric Reconciliation Report.


In [2]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Query benchmark metrics from fact_sales
cur.execute('''
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(DISTINCT Invoice) AS distinct_invoices,
        COUNT(DISTINCT Customer_ID) AS registered_customers,
        SUM(Quantity) AS total_units,
        ROUND(SUM(Revenue), 2) AS gross_revenue
    FROM fact_sales;
''')
sales_bench = cur.fetchone()

# Query benchmark metrics from fact_returns
cur.execute('''
    SELECT 
        COUNT(*) AS return_rows,
        COUNT(DISTINCT Invoice) AS distinct_credit_notes,
        COUNT(DISTINCT Customer_ID) AS return_customers,
        SUM(ABS(Quantity)) AS returned_units,
        ROUND(ABS(SUM(Quantity * Price)), 2) AS total_refund_val
    FROM fact_returns;
''')
returns_bench = cur.fetchone()

bench_df = pd.DataFrame([
    {"Fact Table": "fact_sales", "Total Records": f"{sales_bench[0]:,}", "Distinct Invoices": f"{sales_bench[1]:,}", "Customers": f"{sales_bench[2]:,}", "Units": f"{sales_bench[3]:,}", "Monetary Value": f"£{sales_bench[4]:,.2f}"},
    {"Fact Table": "fact_returns", "Total Records": f"{returns_bench[0]:,}", "Distinct Invoices": f"{returns_bench[1]:,}", "Customers": f"{returns_bench[2]:,}", "Units": f"{returns_bench[3]:,}", "Monetary Value": f"£{returns_bench[4]:,.2f}"}
])

print("--- Source Database Baseline KPIs ---")
display(bench_df)

# Assertions for exact parity
assert sales_bench[0] == 1007983, f"Expected 1,007,983 sales rows, found {sales_bench[0]}"
assert sales_bench[1] == 40083, f"Expected 40,083 distinct invoices, found {sales_bench[1]}"
assert sales_bench[4] == 20476260.45, f"Expected £20,476,260.45 gross revenue, found {sales_bench[4]}"
assert returns_bench[0] == 19104, f"Expected 19,104 return rows, found {returns_bench[0]}"
assert returns_bench[4] == 1462050.61, f"Expected £1,462,050.61 refunds, found {returns_bench[4]}"

print("[PASS] All source database tables perfectly match reconciled benchmarks (100.00% parity).")


--- Source Database Baseline KPIs ---


,Fact Table,Total Records,Distinct Invoices,Customers,Units,Monetary Value
0,fact_sales,"1,007,983","40,083","5,881","11,219,901","£20,476,260.45"
1,fact_returns,"19,104","8,292","2,572","476,821","£1,462,050.61"


[PASS] All source database tables perfectly match reconciled benchmarks (100.00% parity).


## 2. Dimension 1: Centralized Date Dimension (`dim_date`)

A proper Date Dimension is essential for Time Intelligence calculations in Power BI (e.g., `TOTALYTD`, `SAMEPERIODLASTYEAR`, `DATESMTD`).

### Design Specifications
- **Span:** Continuous daily calendar covering every day from `2009-12-01` to `2011-12-31` (761 calendar days).
- **Primary Key:** `Date` (ISO format string `YYYY-MM-DD`), matching foreign keys in fact tables.
- **Attributes:** `DateKey` (integer `YYYYMMDD`), `Year`, `Quarter`, `Quarter_Name`, `Year_Quarter`, `Month`, `Month_Name`, `Month_Short`, `Year_Month`, `Day`, `Day_Name`, `Day_Of_Week` (1=Mon, 7=Sun), `Is_Weekend`, and `Week_Of_Year`.


In [3]:
# Generate continuous calendar range
date_range = pd.date_range(start='2009-12-01', end='2011-12-31', freq='D')

dim_date = pd.DataFrame({'Date_DT': date_range})
dim_date['Date'] = dim_date['Date_DT'].dt.strftime('%Y-%m-%d')
dim_date['DateKey'] = dim_date['Date_DT'].dt.strftime('%Y%m%d').astype(int)
dim_date['Year'] = dim_date['Date_DT'].dt.year
dim_date['Quarter'] = dim_date['Date_DT'].dt.quarter
dim_date['Quarter_Name'] = 'Q' + dim_date['Quarter'].astype(str)
dim_date['Year_Quarter'] = dim_date['Year'].astype(str) + '-Q' + dim_date['Quarter'].astype(str)
dim_date['Month'] = dim_date['Date_DT'].dt.month
dim_date['Month_Name'] = dim_date['Date_DT'].dt.strftime('%B')
dim_date['Month_Short'] = dim_date['Date_DT'].dt.strftime('%b')
dim_date['Year_Month'] = dim_date['Date_DT'].dt.strftime('%Y-%m')
dim_date['Day'] = dim_date['Date_DT'].dt.day
dim_date['Day_Name'] = dim_date['Date_DT'].dt.strftime('%A')
dim_date['Day_Of_Week'] = dim_date['Date_DT'].dt.day_of_week + 1  # 1 = Monday, 7 = Sunday
dim_date['Is_Weekend'] = dim_date['Day_Of_Week'].apply(lambda x: 1 if x in [6, 7] else 0)
dim_date['Week_Of_Year'] = dim_date['Date_DT'].dt.isocalendar().week.astype(int)

# Drop temporary datetime column
dim_date = dim_date.drop(columns=['Date_DT'])

print(f"--- dim_date Generated: {len(dim_date):,} Calendar Days ---")
display(dim_date.head(10))
display(dim_date.tail(5))


--- dim_date Generated: 761 Calendar Days ---


,Date,DateKey,Year,Quarter,Quarter_Name,Year_Quarter,Month,Month_Name,Month_Short,Year_Month,Day,Day_Name,Day_Of_Week,Is_Weekend,Week_Of_Year
0,2009-12-01,20091201,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,1,Tuesday,2,0,49
1,2009-12-02,20091202,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,2,Wednesday,3,0,49
2,2009-12-03,20091203,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,3,Thursday,4,0,49
3,2009-12-04,20091204,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,4,Friday,5,0,49
4,2009-12-05,20091205,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,5,Saturday,6,1,49
5,2009-12-06,20091206,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,6,Sunday,7,1,49
6,2009-12-07,20091207,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,7,Monday,1,0,50
7,2009-12-08,20091208,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,8,Tuesday,2,0,50
8,2009-12-09,20091209,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,9,Wednesday,3,0,50
9,2009-12-10,20091210,2009,4,Q4,2009-Q4,12,December,Dec,2009-12,10,Thursday,4,0,50


,Date,DateKey,Year,Quarter,Quarter_Name,Year_Quarter,Month,Month_Name,Month_Short,Year_Month,Day,Day_Name,Day_Of_Week,Is_Weekend,Week_Of_Year
756,2011-12-27,20111227,2011,4,Q4,2011-Q4,12,December,Dec,2011-12,27,Tuesday,2,0,52
757,2011-12-28,20111228,2011,4,Q4,2011-Q4,12,December,Dec,2011-12,28,Wednesday,3,0,52
758,2011-12-29,20111229,2011,4,Q4,2011-Q4,12,December,Dec,2011-12,29,Thursday,4,0,52
759,2011-12-30,20111230,2011,4,Q4,2011-Q4,12,December,Dec,2011-12,30,Friday,5,0,52
760,2011-12-31,20111231,2011,4,Q4,2011-Q4,12,December,Dec,2011-12,31,Saturday,6,1,52


## 3. Dimension 2: Master Customer Dimension (`dim_customer`)

The Customer Dimension unifies customer attributes, RFM segmentation metrics, historical tenure, and primary geographic residence.

### Referential Integrity Considerations
1. **Active RFM Customers:** 5,881 purchasing accounts categorized into Champions, Loyal Customers, At Risk, etc.
2. **Refund-Only Customers:** 61 accounts that experienced cancellations/refunds but completed no regular sales. To ensure 100% referential integrity in `fact_returns`, they are included with segment label `'Cancellation / Refund Only'`.
3. **Guest Checkouts:** Approximately 15% of transactions have no registered customer identifier. To eliminate referential integrity mismatch on blank values, a designated `'Guest'` record is included.


In [4]:
# 1. Load RFM base table
df_rfm = pd.read_sql("SELECT * FROM dim_customer_rfm", conn)

# 2. Extract customer transaction history dates and primary country from fact_sales
cust_summary_sales = pd.read_sql('''
    SELECT 
        Customer_ID,
        MIN(SUBSTR(InvoiceDate, 1, 10)) AS First_Purchase_Date,
        MAX(SUBSTR(InvoiceDate, 1, 10)) AS Last_Purchase_Date,
        Country AS Customer_Country
    FROM fact_sales
    WHERE Customer_ID IS NOT NULL
    GROUP BY Customer_ID;
''', conn)

# Merge RFM with transaction history
dim_cust_active = pd.merge(df_rfm, cust_summary_sales, on='Customer_ID', how='left')

# 3. Identify and integrate 61 customers appearing only in fact_returns
cust_ret_only = pd.read_sql('''
    SELECT 
        r.Customer_ID,
        COUNT(DISTINCT r.Invoice) AS Frequency,
        SUM(r.Quantity * r.Price) AS Monetary,
        MIN(SUBSTR(r.InvoiceDate, 1, 10)) AS First_Purchase_Date,
        MAX(SUBSTR(r.InvoiceDate, 1, 10)) AS Last_Purchase_Date,
        r.Country AS Customer_Country
    FROM fact_returns r
    LEFT JOIN fact_sales s ON r.Customer_ID = s.Customer_ID
    WHERE s.Customer_ID IS NULL AND r.Customer_ID IS NOT NULL
    GROUP BY r.Customer_ID;
''', conn)

cust_ret_only['Recency'] = 999
cust_ret_only['R_Score'] = 1
cust_ret_only['F_Score'] = 1
cust_ret_only['M_Score'] = 1
cust_ret_only['RFM_Score'] = '111'
cust_ret_only['Segment'] = 'Cancellation / Refund Only'

# 4. Create designated Guest checkout customer entity
guest_entity = pd.DataFrame([{
    'Customer_ID': 'Guest',
    'Recency': 0,
    'Frequency': 0,
    'Monetary': 0.0,
    'R_Score': 0,
    'F_Score': 0,
    'M_Score': 0,
    'RFM_Score': '000',
    'Segment': 'Guest / Unregistered',
    'First_Purchase_Date': '2009-12-01',
    'Last_Purchase_Date': '2011-12-09',
    'Customer_Country': 'Various'
}])

# Combine all customer entities into dim_customer
dim_customer = pd.concat([dim_cust_active, cust_ret_only, guest_entity], ignore_index=True)
dim_customer['Customer_ID'] = dim_customer['Customer_ID'].astype(str)

print(f"--- dim_customer Created: {len(dim_customer):,} Total Entities ---")
print(f"Active RFM Customers:      {len(dim_cust_active):,}")
print(f"Refund-Only Customers:       {len(cust_ret_only):,}")
print(f"Guest Checkout Entity:        1")

print(); print("Customer Segment Breakdown:")
display(dim_customer['Segment'].value_counts().to_frame())
display(dim_customer.head(5))


--- dim_customer Created: 5,943 Total Entities ---
Active RFM Customers:      5,881
Refund-Only Customers:       61
Guest Checkout Entity:        1

Customer Segment Breakdown:


,count
Segment,
Champions,1291
Hibernating,1280
Loyal Customers,1147
At Risk,841
Others,516
Potential Loyalists,415
Cannot Lose Them,224
New Customers,167
Cancellation / Refund Only,61


,Customer_ID,LastPurchaseDate,Frequency,Monetary,TotalItems,Country,Recency,R_Score,F_Score,M_Score,RFM_Score,RFM_Average,Segment,First_Purchase_Date,Last_Purchase_Date,Customer_Country
0,12346,2011-01-18 10:01:00,12,"77,556.46","74,285.00",United Kingdom,325,2,5,5,255,4.00,Cannot Lose Them,2009-12-14,2011-01-18,United Kingdom
1,12347,2011-12-07 15:52:00,8,"4,921.53","2,967.00",Iceland,2,5,4,5,545,4.67,Champions,2010-10-31,2011-12-07,Iceland
2,12348,2011-09-25 13:13:00,5,"2,019.40","2,714.00",Finland,75,3,4,4,344,3.67,Loyal Customers,2010-09-27,2011-09-25,Finland
3,12349,2011-11-21 09:51:00,4,"4,428.69","1,624.00",Italy,18,5,3,5,535,4.33,Loyal Customers,2010-04-29,2011-11-21,Italy
4,12350,2011-02-02 16:01:00,1,334.40,197.00,Norway,310,2,1,2,212,1.67,Hibernating,2011-02-02,2011-02-02,Norway


## 4. Dimension 3: Master Product Dimension (`dim_product`)

The Product Dimension centralizes product SKUs transacted across both commercial sales and return credit notes.

### Product Categorization Taxonomy
To empower high-level merchandise reporting in Power BI, each SKU is classified into one of 8 retail categories based on StockCode prefixes and keyword token parsing:
1. **Home Decor & Lighting:** Lanterns, candles, light holders, wall clocks, signs, frames, buntings.
2. **Kitchen & Tableware:** Mugs, teapots, plates, cake stands, cutlery, jars, glassware.
3. **Bags & Organization:** Jumbo bags, storage boxes, tote bags, lunch bags, purses.
4. **Stationery & Gift Wrap:** Greeting cards, wrapping paper, ribbons, pencils, notebooks.
5. **Holiday & Seasonal:** Christmas decorations, advent calendars, snow globes.
6. **Toys & Children:** Games, dolls, puppets, kids craft sets.
7. **Service / Fee / Postage:** Postage, manual adjustments, bank charges, carrier fees.
8. **General Gifts & Novelties:** General decorative gifts and miscellaneous homeware.


In [5]:
# Aggregate product metrics across fact_sales and fact_returns
sql_product_catalog = '''
SELECT 
    p.StockCode,
    MAX(p.Description) AS Description,
    SUM(p.units_sold) AS total_units_sold,
    ROUND(SUM(p.revenue), 2) AS total_gross_revenue,
    SUM(p.orders) AS total_orders,
    ROUND(AVG(CASE WHEN p.price > 0 THEN p.price END), 2) AS avg_price,
    SUM(p.units_refunded) AS total_units_refunded,
    ROUND(SUM(p.refund_amt), 2) AS total_refund_amount,
    SUM(p.return_orders) AS total_return_orders
FROM (
    SELECT 
        StockCode, 
        Description, 
        Quantity AS units_sold, 
        Revenue AS revenue, 
        1 AS orders, 
        Price AS price, 
        0 AS units_refunded, 
        0.0 AS refund_amt, 
        0 AS return_orders
    FROM fact_sales
    
    UNION ALL
    
    SELECT 
        StockCode, 
        Description, 
        0 AS units_sold, 
        0.0 AS revenue, 
        0 AS orders, 
        Price AS price, 
        ABS(Quantity) AS units_refunded, 
        ABS(Quantity * Price) AS refund_amt, 
        1 AS return_orders
    FROM fact_returns
) p
GROUP BY p.StockCode;
'''

dim_product = pd.read_sql(sql_product_catalog, conn)
dim_product['StockCode'] = dim_product['StockCode'].astype(str)

# Categorization logic
def assign_product_category(row):
    sc = str(row['StockCode']).strip().upper()
    desc = str(row['Description']).strip().upper()
    if sc in ['POST', 'DOT', 'M', 'D', 'CRUK', 'BANK CHARGES', 'AMAZONFEE']:
        return 'Service / Fee / Postage'
    elif any(k in desc for k in ['CHRISTMAS', 'XMAS', 'ADVENT', 'SNOW']):
        return 'Holiday & Seasonal'
    elif any(k in desc for k in ['HEART', 'LIGHT', 'CANDLE', 'LANTERN', 'HANGER', 'CLOCK', 'SIGN', 'FRAME', 'BUNTING']):
        return 'Home Decor & Lighting'
    elif any(k in desc for k in ['MUG', 'CUP', 'TEA', 'BOWL', 'PLATE', 'CUTLERY', 'JAR', 'CAKE', 'BOTTLE']):
        return 'Kitchen & Tableware'
    elif any(k in desc for k in ['BAG', 'BOX', 'STORAGE', 'TOTE', 'PURSE', 'WALLET']):
        return 'Bags & Organization'
    elif any(k in desc for k in ['CARD', 'WRAP', 'PAPER', 'RIBBON', 'PENCIL', 'PEN', 'NOTEBOOK']):
        return 'Stationery & Gift Wrap'
    elif any(k in desc for k in ['TOY', 'GAME', 'DOLL', 'CHILD', 'KIDS', 'PUPPET']):
        return 'Toys & Children'
    else:
        return 'General Gifts & Novelties'

dim_product['Category'] = dim_product.apply(assign_product_category, axis=1)

print(f"--- dim_product Created: {len(dim_product):,} Total Product SKUs ---")
print(); print("Product Taxonomy Distribution:")
display(dim_product['Category'].value_counts().to_frame())
display(dim_product.head(5))


--- dim_product Created: 4,932 Total Product SKUs ---

Product Taxonomy Distribution:


,count
Category,
General Gifts & Novelties,2211
Home Decor & Lighting,1172
Kitchen & Tableware,508
Stationery & Gift Wrap,370
Bags & Organization,353
Holiday & Seasonal,214
Toys & Children,96
Service / Fee / Postage,8


,StockCode,Description,total_units_sold,total_gross_revenue,total_orders,avg_price,total_units_refunded,total_refund_amount,total_return_orders,Category
0,10002,INFLATABLE POLITICAL GLOBE,8671,"6,942.26",367,0.98,1223,883.55,7,General Gifts & Novelties
1,10002R,ROBOT PENCIL SHARPNER,4,20.57,3,5.13,0,0.00,0,Stationery & Gift Wrap
2,10080,GROOVY CACTUS INFLATABLE,315,129.29,28,0.51,0,0.00,0,General Gifts & Novelties
3,10109,BENDY COLOUR PENCILS,4,1.68,1,0.42,0,0.00,0,Stationery & Gift Wrap
4,10120,DOGGY RUBBER,664,142.74,73,0.24,30,6.30,1,General Gifts & Novelties


## 5. Dimension 4: Geographic Dimension (`dim_country`)

The Country Dimension maps geographic sales performance, order counts, registered buyers, and regional market clustering.

### Geographic Grouping & Market Tiers
- **Domestic (UK):** Tier 1 market generating £17.41M (85.03% of gross sales).
- **Western Europe:** Tier 2 markets (EIRE, Netherlands, Germany, France) each generating £300k–£700k.
- **Developed Export Markets:** Tier 3 international markets (Australia, Spain, Switzerland, Sweden, Denmark, etc.).
- **Rest of World / Emerging:** Tier 4 markets with emerging commercial adoption.


In [6]:
sql_country_catalog = '''
SELECT 
    Country,
    COUNT(DISTINCT Invoice) AS total_orders,
    COUNT(DISTINCT Customer_ID) AS registered_customers,
    SUM(Quantity) AS total_units_sold,
    ROUND(SUM(Revenue), 2) AS gross_revenue,
    ROUND(SUM(Revenue) / COUNT(DISTINCT Invoice), 2) AS aov
FROM fact_sales
GROUP BY Country;
'''

dim_country = pd.read_sql(sql_country_catalog, conn)

def assign_country_tier(c):
    if c == 'United Kingdom':
        return 'Domestic (UK)', 'Tier 1 (>£10M)'
    elif c in ['EIRE', 'Netherlands', 'Germany', 'France']:
        return 'Western Europe', 'Tier 2 (£200k-£1M)'
    elif c in ['Australia', 'Spain', 'Switzerland', 'Sweden', 'Denmark', 'Belgium', 'Channel Islands', 'Norway', 'Portugal', 'Japan']:
        return 'Developed Export Markets', 'Tier 3 (£20k-£200k)'
    else:
        return 'Rest of World / Emerging', 'Tier 4 (<£20k)'

tier_assignments = [assign_country_tier(c) for c in dim_country['Country']]
dim_country['Region'] = [t[0] for t in tier_assignments]
dim_country['Market_Tier'] = [t[1] for t in tier_assignments]

print(f"--- dim_country Created: {len(dim_country):,} Transacted Countries ---")
display(dim_country.sort_values(by='gross_revenue', ascending=False).head(10))


--- dim_country Created: 43 Transacted Countries ---


,Country,total_orders,registered_customers,total_units_sold,gross_revenue,aov,Region,Market_Tier
40,United Kingdom,36540,5353,9201263,"17,410,196.12",476.47,Domestic (UK),Tier 1 (>£10M)
11,EIRE,626,5,336579,"658,767.31","1,052.34",Western Europe,Tier 2 (£200k-£1M)
26,Netherlands,229,22,384519,"554,038.09","2,419.38",Western Europe,Tier 2 (£200k-£1M)
15,Germany,789,107,225173,"425,019.71",538.68,Western Europe,Tier 2 (£200k-£1M)
14,France,622,95,271902,"350,456.09",563.43,Western Europe,Tier 2 (£200k-£1M)
0,Australia,95,15,104067,"169,283.46","1,781.93",Developed Export Markets,Tier 3 (£20k-£200k)
34,Spain,154,41,50318,"108,332.49",703.46,Developed Export Markets,Tier 3 (£20k-£200k)
36,Switzerland,93,22,52763,"100,685.59","1,082.64",Developed Export Markets,Tier 3 (£20k-£200k)
35,Sweden,105,19,88633,"91,869.82",874.95,Developed Export Markets,Tier 3 (£20k-£200k)
10,Denmark,43,12,237471,"68,580.69","1,594.90",Developed Export Markets,Tier 3 (£20k-£200k)


## 6. Fact Table Optimization (`fact_sales` & `fact_returns`)

To construct a high-performance Star Schema, transaction fact tables are optimized:
1. **Remove Denormalized Calendar Fields:** `Year`, `Month`, `Quarter`, `Day`, and `Day_Name` are stripped from fact tables; all temporal slicing is performed through `dim_date`.
2. **Explicit Foreign Key Alignment:** A clean `Date` key (`YYYY-MM-DD`) is extracted to join directly with `dim_date[Date]`.
3. **Explicit Guest Key Assignment:** Unassigned `Customer_ID` values are populated with `'Guest'`, perfectly matching the `'Guest'` record in `dim_customer`.
4. **Dual Refund Measures:** `fact_returns` provides both signed `Refund_Amount` (negative for net financial calculations) and `Abs_Refund_Amount` (positive for visual volume charts).


In [7]:
# Optimize fact_sales
fact_sales_optimized = pd.read_sql('''
SELECT 
    Invoice,
    StockCode,
    InvoiceDate,
    SUBSTR(InvoiceDate, 1, 10) AS Date,
    Quantity,
    Price,
    Revenue,
    COALESCE(Customer_ID, 'Guest') AS Customer_ID,
    Country
FROM fact_sales;
''', conn)

fact_sales_optimized['Invoice'] = fact_sales_optimized['Invoice'].astype(str)
fact_sales_optimized['StockCode'] = fact_sales_optimized['StockCode'].astype(str)
fact_sales_optimized['Customer_ID'] = fact_sales_optimized['Customer_ID'].astype(str)

# Optimize fact_returns
fact_returns_optimized = pd.read_sql('''
SELECT 
    Invoice,
    StockCode,
    InvoiceDate,
    SUBSTR(InvoiceDate, 1, 10) AS Date,
    Quantity,
    Price,
    ROUND(Quantity * Price, 2) AS Refund_Amount,
    ABS(ROUND(Quantity * Price, 2)) AS Abs_Refund_Amount,
    COALESCE(Customer_ID, 'Guest') AS Customer_ID,
    Country
FROM fact_returns;
''', conn)

fact_returns_optimized['Invoice'] = fact_returns_optimized['Invoice'].astype(str)
fact_returns_optimized['StockCode'] = fact_returns_optimized['StockCode'].astype(str)
fact_returns_optimized['Customer_ID'] = fact_returns_optimized['Customer_ID'].astype(str)

print(f"--- Fact Tables Optimized ---")
print(f"fact_sales:   {fact_sales_optimized.shape[0]:,} rows | {fact_sales_optimized.shape[1]} columns")
print(f"fact_returns:  {fact_returns_optimized.shape[0]:,} rows | {fact_returns_optimized.shape[1]} columns")
display(fact_sales_optimized.head(3))
display(fact_returns_optimized.head(3))


--- Fact Tables Optimized ---
fact_sales:   1,007,983 rows | 9 columns
fact_returns:  19,104 rows | 10 columns


,Invoice,StockCode,InvoiceDate,Date,Quantity,Price,Revenue,Customer_ID,Country
0,489434,85048,2009-12-01 07:45:00,2009-12-01,12,6.95,83.40,13085,United Kingdom
1,489434,79323P,2009-12-01 07:45:00,2009-12-01,12,6.75,81.00,13085,United Kingdom
2,489434,79323W,2009-12-01 07:45:00,2009-12-01,12,6.75,81.00,13085,United Kingdom


,Invoice,StockCode,InvoiceDate,Date,Quantity,Price,Refund_Amount,Abs_Refund_Amount,Customer_ID,Country
0,C489449,22087,2009-12-01 10:33:00,2009-12-01,-12,2.95,-35.40,35.40,16321,Australia
1,C489449,85206A,2009-12-01 10:33:00,2009-12-01,-6,1.65,-9.90,9.90,16321,Australia
2,C489449,21895,2009-12-01 10:33:00,2009-12-01,-4,4.25,-17.00,17.00,16321,Australia


## 7. Comprehensive Referential Integrity & Zero-Orphan Key Audit

We execute strict referential integrity validation across every foreign key in both fact tables to guarantee zero orphan records prior to loading into Power BI.


In [8]:
# Validate fact_sales foreign keys
fs_invalid_date = (~fact_sales_optimized['Date'].isin(dim_date['Date'])).sum()
fs_invalid_cust = (~fact_sales_optimized['Customer_ID'].isin(dim_customer['Customer_ID'])).sum()
fs_invalid_prod = (~fact_sales_optimized['StockCode'].isin(dim_product['StockCode'])).sum()
fs_invalid_ctry = (~fact_sales_optimized['Country'].isin(dim_country['Country'])).sum()

# Validate fact_returns foreign keys
fr_invalid_date = (~fact_returns_optimized['Date'].isin(dim_date['Date'])).sum()
fr_invalid_cust = (~fact_returns_optimized['Customer_ID'].isin(dim_customer['Customer_ID'])).sum()
fr_invalid_prod = (~fact_returns_optimized['StockCode'].isin(dim_product['StockCode'])).sum()
fr_invalid_ctry = (~fact_returns_optimized['Country'].isin(dim_country['Country'])).sum()

audit_results = [
    {"Fact Table": "fact_sales", "FK Column": "Date", "Parent Dimension": "dim_date[Date]", "Orphan Rows": fs_invalid_date, "Audit Status": "PASS" if fs_invalid_date == 0 else "FAIL"},
    {"Fact Table": "fact_sales", "FK Column": "Customer_ID", "Parent Dimension": "dim_customer[Customer_ID]", "Orphan Rows": fs_invalid_cust, "Audit Status": "PASS" if fs_invalid_cust == 0 else "FAIL"},
    {"Fact Table": "fact_sales", "FK Column": "StockCode", "Parent Dimension": "dim_product[StockCode]", "Orphan Rows": fs_invalid_prod, "Audit Status": "PASS" if fs_invalid_prod == 0 else "FAIL"},
    {"Fact Table": "fact_sales", "FK Column": "Country", "Parent Dimension": "dim_country[Country]", "Orphan Rows": fs_invalid_ctry, "Audit Status": "PASS" if fs_invalid_ctry == 0 else "FAIL"},
    {"Fact Table": "fact_returns", "FK Column": "Date", "Parent Dimension": "dim_date[Date]", "Orphan Rows": fr_invalid_date, "Audit Status": "PASS" if fr_invalid_date == 0 else "FAIL"},
    {"Fact Table": "fact_returns", "FK Column": "Customer_ID", "Parent Dimension": "dim_customer[Customer_ID]", "Orphan Rows": fr_invalid_cust, "Audit Status": "PASS" if fr_invalid_cust == 0 else "FAIL"},
    {"Fact Table": "fact_returns", "FK Column": "StockCode", "Parent Dimension": "dim_product[StockCode]", "Orphan Rows": fr_invalid_prod, "Audit Status": "PASS" if fr_invalid_prod == 0 else "FAIL"},
    {"Fact Table": "fact_returns", "FK Column": "Country", "Parent Dimension": "dim_country[Country]", "Orphan Rows": fr_invalid_ctry, "Audit Status": "PASS" if fr_invalid_ctry == 0 else "FAIL"}
]

audit_df = pd.DataFrame(audit_results)
print("==========================================================================")
print("REFERENTIAL INTEGRITY & ZERO-ORPHAN KEY AUDIT REPORT")
print("==========================================================================")
display(audit_df)

assert audit_df['Orphan Rows'].sum() == 0, "Referential integrity failure: orphan keys detected!"
print(); print("[SUCCESS] 100.00% Referential Integrity Verified: Zero orphan keys across all dimensions.")


REFERENTIAL INTEGRITY & ZERO-ORPHAN KEY AUDIT REPORT


,Fact Table,FK Column,Parent Dimension,Orphan Rows,Audit Status
0,fact_sales,Date,dim_date[Date],0,PASS
1,fact_sales,Customer_ID,dim_customer[Customer_ID],0,PASS
2,fact_sales,StockCode,dim_product[StockCode],0,PASS
3,fact_sales,Country,dim_country[Country],0,PASS
4,fact_returns,Date,dim_date[Date],0,PASS
5,fact_returns,Customer_ID,dim_customer[Customer_ID],0,PASS
6,fact_returns,StockCode,dim_product[StockCode],0,PASS
7,fact_returns,Country,dim_country[Country],0,PASS



[SUCCESS] 100.00% Referential Integrity Verified: Zero orphan keys across all dimensions.


## 8. Database Persistence & Power BI File Exports

All validated dimension tables are persisted into SQLite (`data/processed/ecommerce_analytics.db`) and exported as high-speed CSV datasets in `data/processed/powerbi/`.


In [9]:
# 1. Persist Dimensions into SQLite database
print("Saving dimensional tables into SQLite database...")
dim_date.to_sql('dim_date', conn, if_exists='replace', index=False)
dim_customer.to_sql('dim_customer', conn, if_exists='replace', index=False)
dim_product.to_sql('dim_product', conn, if_exists='replace', index=False)
dim_country.to_sql('dim_country', conn, if_exists='replace', index=False)

# Create index on primary keys in SQLite
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_dim_date_pk ON dim_date(Date);")
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_dim_cust_pk ON dim_customer(Customer_ID);")
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_dim_prod_pk ON dim_product(StockCode);")
cur.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_dim_ctry_pk ON dim_country(Country);")
conn.commit()
print("[OK] SQLite dimensional tables and primary key indexes saved successfully.")

# 2. Export clean CSV datasets for direct Power BI Desktop ingestion
print(); print("Exporting datasets to data/processed/powerbi/...")
dim_date.to_csv(os.path.join(pbi_dir, 'dim_date.csv'), index=False)
dim_customer.to_csv(os.path.join(pbi_dir, 'dim_customer.csv'), index=False)
dim_product.to_csv(os.path.join(pbi_dir, 'dim_product.csv'), index=False)
dim_country.to_csv(os.path.join(pbi_dir, 'dim_country.csv'), index=False)
fact_sales_optimized.to_csv(os.path.join(pbi_dir, 'fact_sales.csv'), index=False)
fact_returns_optimized.to_csv(os.path.join(pbi_dir, 'fact_returns.csv'), index=False)

# File size summary
file_stats = []
for fname in ['dim_date.csv', 'dim_customer.csv', 'dim_product.csv', 'dim_country.csv', 'fact_sales.csv', 'fact_returns.csv']:
    fpath = os.path.join(pbi_dir, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    file_stats.append({"File Name": fname, "Size (MB)": f"{size_mb:.2f} MB"})

display(pd.DataFrame(file_stats))
print("[OK] All Power BI source files successfully exported.")


Saving dimensional tables into SQLite database...


[OK] SQLite dimensional tables and primary key indexes saved successfully.

Exporting datasets to data/processed/powerbi/...


,File Name,Size (MB)
0,dim_date.csv,0.06 MB
1,dim_customer.csv,0.71 MB
2,dim_product.csv,0.42 MB
3,dim_country.csv,0.00 MB
4,fact_sales.csv,76.97 MB
5,fact_returns.csv,1.55 MB


[OK] All Power BI source files successfully exported.


## 9. Power BI Data Model Architecture Summary

### Prepared Star Schema Matrix
- **`dim_date`** (761 rows): Marked Date table for temporal analysis and DAX time intelligence.
- **`dim_customer`** (5,943 rows): Master customer profiles with RFM segments, scores, and guest entity.
- **`dim_product`** (4,932 rows): Complete product catalog with unit pricing and retail merchandise taxonomy.
- **`dim_country`** (43 rows): Global country dimensions with regional groupings and market revenue tiers.
- **`fact_sales`** (1,007,983 rows): Validated commercial transactions (£20,476,260.45 gross revenue).
- **`fact_returns`** (19,104 rows): Credit note cancellations (-£1,462,050.61 refunds).

The semantic layer is 100% prepared for Power BI Desktop data modeling and dashboard creation.


In [10]:
# Close database connection
conn.close()

summary_metrics = pd.DataFrame([
    {"Table": "dim_date", "Type": "Dimension", "PK / FK": "PK: Date", "Rows": f"{len(dim_date):,}", "Primary Use": "Temporal Slicing & DAX Time Intelligence"},
    {"Table": "dim_customer", "Type": "Dimension", "PK / FK": "PK: Customer_ID", "Rows": f"{len(dim_customer):,}", "Primary Use": "RFM Segmentation & Customer Lifetime Analysis"},
    {"Table": "dim_product", "Type": "Dimension", "PK / FK": "PK: StockCode", "Rows": f"{len(dim_product):,}", "Primary Use": "Merchandise Category & SKU Intelligence"},
    {"Table": "dim_country", "Type": "Dimension", "PK / FK": "PK: Country", "Rows": f"{len(dim_country):,}", "Primary Use": "Geographic Sales & Export Market Analysis"},
    {"Table": "fact_sales", "Type": "Fact", "PK / FK": "FKs: Date, Customer, SKU, Country", "Rows": f"{len(fact_sales_optimized):,}", "Primary Use": "Gross Commercial Sales (£20.48M)"},
    {"Table": "fact_returns", "Type": "Fact", "PK / FK": "FKs: Date, Customer, SKU, Country", "Rows": f"{len(fact_returns_optimized):,}", "Primary Use": "Customer Refunds & Net Sales Auditing (£1.46M)"}
])

print("==========================================================================")
print("ENTERPRISE STAR SCHEMA ARCHITECTURE READY FOR POWER BI")
print("==========================================================================")
display(summary_metrics)
print(); print("[READY] Phase 7 Data Preparation Complete. All source tables validated and exported.")


ENTERPRISE STAR SCHEMA ARCHITECTURE READY FOR POWER BI


,Table,Type,PK / FK,Rows,Primary Use
0,dim_date,Dimension,PK: Date,761,Temporal Slicing & DAX Time Intelligence
1,dim_customer,Dimension,PK: Customer_ID,"5,943",RFM Segmentation & Customer Lifetime Analysis
2,dim_product,Dimension,PK: StockCode,"4,932",Merchandise Category & SKU Intelligence
3,dim_country,Dimension,PK: Country,43,Geographic Sales & Export Market Analysis
4,fact_sales,Fact,"FKs: Date, Customer, SKU, Country","1,007,983",Gross Commercial Sales (£20.48M)
5,fact_returns,Fact,"FKs: Date, Customer, SKU, Country","19,104",Customer Refunds & Net Sales Auditing (£1.46M)



[READY] Phase 7 Data Preparation Complete. All source tables validated and exported.
